# Example: Facebook

In [21]:
import pandas as pd
import pandas_datareader.data as web 
import datetime
import yfinance as yf

import statsmodels.api as sm 

import numpy as np 
import matplotlib.pyplot as plt


In [22]:
#import datetime imports the module named "datetime"
#Also a class inside called datetime hence need twice
from datetime import datetime 
# alternative is datetime.datetime...
# (module.class)

start = datetime(2020,6,28)
end = datetime(2025,6,28)

In [23]:
#Downloading the data 
#DataReader's signature is DataReader(name, data_source, start, end)
#hence 'fred' is required 

sp500 = web.DataReader(['sp500'],'fred',start,end)
stock = yf.download('META',start,end)

[*********************100%***********************]  1 of 1 completed


In [24]:
#.columns, .head, .shape can be used to investigate

stock.head()


Price,Close,High,Low,Open,Volume
Ticker,META,META,META,META,META
Date,,,,,
2020-06-29,218.719315,218.828358,205.307095,207.924113,58514300
2020-06-30,225.093338,225.519588,216.568196,218.669736,33927100
2020-07-01,235.482117,236.919491,225.579075,226.510894,43399700
2020-07-02,231.388046,237.910768,230.585100,236.919473,30633600
2020-07-06,238.188324,238.307275,230.248058,231.725077,26206200


In [25]:
sp500.columns


Index(['sp500'], dtype='object')

In [26]:
#combining two datasets 

#can't use () as pandas will ook for single column named ('Date','Close')
#double brackets is a list containing one item and that item is a list itself


sp500 = sp500.reset_index() 
stock = stock.reset_index()

stock.columns = stock.columns.droplevel(1) #Drop the 'META' ticker level

In [27]:
#creates new dataframe containing the date column and the close column - full columns

stock = stock[['Date','Close']] #decides which columns of data survive
stock.columns = ['Date', 'stock'] #renaming column's that survived 

#renaming the stock's price column to stock now means after the merge there will be three cleanly named columns
sp500.columns = ['Date', 'sp500'] 

#stock (Date + stock price) and sp500 (Date + index level)
#merge joins them together by matching rows that share a common key, which here is the date 

#on= matches on a column, and both must hae one with that exact name
data = pd.merge(stock,sp500,on='Date')


In [28]:
data

,Date,stock,sp500
0,2020-06-29,218.719315,3053.24
1,2020-06-30,225.093338,3100.29
2,2020-07-01,235.482117,3115.86
3,2020-07-02,231.388046,3130.01
4,2020-07-06,238.188324,3179.72
...,...,...,...
1251,2025-06-23,696.248047,6025.17
1252,2025-06-24,709.873413,6092.18
1253,2025-06-25,706.364868,6092.16
1254,2025-06-26,723.718018,6141.02


In [29]:
# .iloc - integer by position, "give me by position, 0-based, like a pyhton list"
# .loc - by label. "Give me by the label on the index/columns", not position

#creating a new column in data and label
#when assigning to a column label that doesn't exist yet it creates it.
#same syntax creates and overrides what may already be there, hence dats is now 5 columns

data['sp500_ret'] = data.loc[:, 'sp500'].pct_change() #can use data['stock'] instead as well 
data['stock_ret'] = data.loc[:,'stock'].pct_change()

final_data = data.iloc[1:]

x = final_data['sp500_ret'] #market returns
y = final_data ['stock_ret'] #stock returns 

final_data.head()

,Date,stock,sp500,sp500_ret,stock_ret
1,2020-06-30,225.093338,3100.29,0.015410,0.029142
2,2020-07-01,235.482117,3115.86,0.005022,0.046153
3,2020-07-02,231.388046,3130.01,0.004541,-0.017386
4,2020-07-06,238.188324,3179.72,0.015882,0.029389
5,2020-07-07,238.763275,3145.32,-0.010819,0.002414


In [30]:
x1 = sm.add_constant(x) #adding the intercept column
model = sm.OLS(y, x1) #dependenet variable first. Sklearn is opposite order.
results = model.fit() #runs the fit 
print(results.summary())

                            OLS Regression Results                            
Dep. Variable:              stock_ret   R-squared:                       0.401
Model:                            OLS   Adj. R-squared:                  0.400
Method:                 Least Squares   F-statistic:                     838.7
Date:                Mon, 31 Aug 2026   Prob (F-statistic):          1.34e-141
Time:                        18:38:48   Log-Likelihood:                 3042.9
No. Observations:                1255   AIC:                            -6082.
Df Residuals:                    1253   BIC:                            -6072.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0004      0.001      0.590      0.5

In [31]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import r2_score


In [32]:
#x.reshape(-1,1) - the -1 means "you can work out how many rows, just make it exactly 1 column"
# reshape is a numpy method hence needs to be in numpy, earlier selected the single colum with single brackets hence single brackets return pandas series not numpy array 
#hence convert to numpy 

import numpy as np 
x = np.array(x).reshape(-1,1)
x

#double brackts, the outer [] is the whole array; each inner [] wraps a singhle number on its own row. That nesting is the 2D structure


array([[ 1.54098597e-02],
       [ 5.02211083e-03],
       [ 4.54128234e-03],
       ...,
       [-3.28289709e-06],
       [ 8.02014392e-03],
       [ 5.21900271e-03]])

In [33]:
#now the data split, holding out the test set
#train_test_split runs once and returns 4 things, the splitting isn't happening to all of them
#N inputs and N outputs then python lines them up.
#If the RHS produces a bundle sof N things, you can put N names on the left , separated by commas, and python assigns them position by position in one go 
#hence important to keep the ordering correct 
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.3,random_state=42)

print(x_train.shape,x_test.shape,y_train.shape,y_test.shape)

#X is a 2D Array whilse y is a 1D array. X is the feature hence must be 2D whilst y is the target, stays 1D

(878, 1) (377, 1) (878,) (377,)


Fit high-order polynomial features to illustrate overfitting.

Note that PolynomialFeatures from sklearn automatically includes an intercept term in its output (a column of 1s as the first column).

make_pipeline() method is used to create a pipeline which first executes PolynomialFeatures and then LinearRegression model.



In [34]:
#PolynomialFeatures(degree=d) takes single feature x and expands it into 1,x,x^2,...,x^d.
#Higher degrees = more flexability to "wiggle"
#make_pipeline(A,B) chains steps so they run in sequence: it feeds data through A and then B as one combined model
# means "first expand x into polynomial features, then fit a linear regression on those features."
#pipeline handles passing the output of step 1 into step 2. So you .fit() and .predict() on the whole thing as if single model 

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import r2_score

#need for loop to run each degree one at a time

degrees = [1,3,5,10,15] #list defined outside of loop 
for degree in degrees:

#Build pipeline PolynomialFeatures(degree)->LinearRegression
#Fit on x_train and y_train, every sklearn model fits with .fit(features,target)
    model = make_pipeline(PolynomialFeatures(degree=degree), LinearRegression())
    model.fit(x_train,y_train)

    y_train_pred = model.predict(x_train)
    y_test_pred = model.predict(x_test)

    r2_train = r2_score(y_train,y_train_pred)
    r2_test = r2_score(y_test,y_test_pred)

    print(f"degree {degree}: train R2 {r2_train:.4f}, test R2 {r2_test:.4f}")

#Comparing x to y - market returns against stock returns directly. r2_score needs to compare the true y against the model's predicted y


degree 1: train R2 0.4123, test R2 0.3704
degree 3: train R2 0.4137, test R2 0.3696
degree 5: train R2 0.4158, test R2 0.3674
degree 10: train R2 0.4163, test R2 0.3669
degree 15: train R2 0.4163, test R2 0.3669


In [35]:
from sklearn.linear_model import Ridge 

# Model using Ridge Regression 

alphas = [0,0.001,0.01,1,10,100]
degree = 10

for alpha in alphas:
    model = make_pipeline(PolynomialFeatures(degree = degree),Ridge(alpha=alpha))
    model.fit(x_train,y_train)

    y_train_pred = model.predict(x_train)
    y_test_pred = model.predict(x_test)

    r2_train = r2_score(y_train,y_train_pred)
    r2_test = r2_score(y_test,y_test_pred)

    print(f"degree {degree}, {alpha}: train R2 {r2_train:.4f}, test R2 {r2_test:.4f}")
    print(f"R2 Gap = {r2_train - r2_test}")
    
# Increasing alpha penalises large coeffs more


degree 10, 0: train R2 0.4195, test R2 0.3377
R2 Gap = 0.08173486357605297
degree 10, 0.001: train R2 0.4123, test R2 0.3705
R2 Gap = 0.041789738794938414
degree 10, 0.01: train R2 0.4093, test R2 0.3688
R2 Gap = 0.04048601016493225
degree 10, 1: train R2 0.0766, test R2 0.0660
R2 Gap = 0.010635294577145404
degree 10, 10: train R2 0.0088, test R2 0.0031
R2 Gap = 0.005709724804914051
degree 10, 100: train R2 0.0009, test R2 -0.0043
R2 Gap = 0.0051437232029073465


In [36]:
#LASSO 

from sklearn.linear_model import Lasso 

alphas = [0.0001, 0.001, 0.01, 0.1, 1]
degree = 10

for alpha in alphas:
    model = make_pipeline(PolynomialFeatures(degree=degree), Lasso(alpha=alpha))
    model.fit(x_train,y_train)

    y_train_pred = model.predict(x_train)
    y_test_pred = model.predict(x_test)

    r2_train = r2_score(y_train,y_train_pred)
    r2_test = r2_score(y_test,y_test_pred)

    coefs = model.named_steps['lasso'].coef_
    n_nonzero = (coefs != 0).sum()
    print(f"alpha {alpha}: test R2 {r2_test:.4f}, non-zero {n_nonzero}/{len(coefs)}")


    
                    


alpha 0.0001: test R2 0.2781, non-zero 1/11
alpha 0.001: test R2 -0.0051, non-zero 0/11
alpha 0.01: test R2 -0.0051, non-zero 0/11
alpha 0.1: test R2 -0.0051, non-zero 0/11
alpha 1: test R2 -0.0051, non-zero 0/11


In [61]:
from sklearn.model_selection import TimeSeriesSplit 
from sklearn.linear_model import Lasso
import numpy as np 
tscv = TimeSeriesSplit(n_splits=5)

#y is still a pandas Series whose index starts at 1 (we dropped row 0 earlier)
#y[train_index] would look up LABELS 0,1,2... and crash on label 0, so turn it into a plain numpy array first
y = np.array(y)


for train_index, test_index in tscv.split(x):
    x_train, x_test = x[train_index], x[test_index] 
    y_train, y_test = y[train_index], y[test_index]

    model = LinearRegression()
    model.fit(x_train,y_train)

    y_test_pred = model.predict(x_test)
    r2_test = r2_score(y_test, y_test_pred)

    print(f"test R2 {r2_test:.4f}")





test R2 0.3282
test R2 0.4424
test R2 0.3171
test R2 0.2636
test R2 0.6076


# Example: What Is Your Favorite Stock?

In [ ]:
#same steps as the META example, wrapped in a function so any ticker can be checked in one line

def estimate_beta(ticker, start=start, end=end):
    stock = yf.download(ticker, start, end, progress=False)
    stock.columns = stock.columns.droplevel(1) #drop the ticker level like before
    stock = stock.reset_index()[['Date', 'Close']]
    stock.columns = ['Date', 'stock']

    market = web.DataReader(['sp500'], 'fred', start, end).reset_index()
    market.columns = ['Date', 'sp500']

    df = pd.merge(stock, market, on='Date')
    df['stock_ret'] = df['stock'].pct_change()
    df['sp500_ret'] = df['sp500'].pct_change()
    df = df.dropna() #FRED has blank days (holidays) which would break the regression

    res = sm.OLS(df['stock_ret'], sm.add_constant(df['sp500_ret'])).fit()
    #params[1] is the slope = beta, params[0] the intercept = daily alpha
    return res.params.iloc[1], res.rsquared

#a few stocks with very different businesses
#expect: utilities and consumer staples low beta, tech and banks higher
for ticker in ['META', 'NVDA', 'JPM', 'KO', 'DUK']:
    beta, r2 = estimate_beta(ticker)
    print(f"{ticker}: beta {beta:.2f}, R2 {r2:.2f}")
